#### ■事前作業
- 指定のダッシュボードからデータをダウンロードして、このファイルと同じフォルダに「raw.csv」と名前を付けて保存する。 …… ダウンロードデータ (1)
- 指定のダッシュボードからデータをダウンロードして、このファイルと同じフォルダに「reg.xlsx」と名前を付けて保存する。 …… ダウンロードデータ (2)

In [2]:
import pandas as pd
import duckdb
import os
import time
from openpyxl.utils import get_column_letter
import webbrowser

In [ ]:
# ファイルのアップロード先を既定のブラウザで開いておく
url_upload = 'https://～～～アップロード先～～～'
webbrowser.open(url_upload)

In [4]:
# ------------------------------------------
# DuckDBとの接続（セッション）を作る
# ------------------------------------------
con = duckdb.connect()

# ------------------------------------------
# ダウンロードデータ (1)
# ------------------------------------------
con.sql("""
    DROP TABLE IF EXISTS raw;
    CREATE TEMP TABLE raw AS
    SELECT *
    FROM 'raw.csv'
""")

df_raw = con.sql("""
    SELECT *
    FROM raw
""").df()

df_raw.head(3)

,datetime,region,progress,id,type,judgment_criteria,alphabet,iroha
0,2026-06-15,東海,90,abc111,TIGER,1163,ABC,いろは
1,2026-06-16,関東,90,abc112,TIGER,324,ABC,いろは
2,2026-06-17,関西,10,abc113,TIGER,37,ABC,いろは


In [5]:
# ------------------------------------------
# テストアカウントの定義（除外するID）
# ------------------------------------------
df_test_account = pd.read_excel("test_account.xlsx")

con.sql("""
    DROP TABLE IF EXISTS test_account;
""")

con.register("test_account", df_test_account)

df_test_account.head(3)

,id,name
0,abc129,Test1
1,abc130,Test2
2,abc131,Test3


In [6]:
# ------------------------------------------
# ダウンロードデータ (1) の編集
# ------------------------------------------
con.sql("""
    DROP TABLE IF EXISTS modified;
    CREATE TEMP TABLE modified AS
    SELECT
        raw.*
        , CASE WHEN judgment_criteria >= 10000 THEN 'S'
               WHEN judgment_criteria >=  5000 THEN 'A'
               WHEN judgment_criteria >=  2500 THEN 'B'
               WHEN judgment_criteria >=  1000 THEN 'C'
               WHEN judgment_criteria >=   500 THEN 'D'
               WHEN judgment_criteria >=   100 THEN 'E'
               ELSE 'F'
          END AS tier
    FROM raw
    WHERE
        COALESCE(progress, 0) < 80
        AND UPPER(type) IN ('LION', 'TIGER', 'EAGLE', 'HAWK')
        AND id NOT IN (SELECT id FROM test_account)
""")

df_modified = con.sql("""
    SELECT *
    FROM modified
""").df()

df_modified = df_modified.sort_values(['judgment_criteria'], ascending=[False], na_position='last')
df_modified['id'] = df_modified['id'].astype(str)
df_modified.to_excel("temp_modified.xlsx", sheet_name="Sheet1", index=False) # 処理過程確認用

df_modified.head(3)

,datetime,region,progress,id,type,judgment_criteria,alphabet,iroha,tier
21,2026-07-28,中部,<NA>,abc154,EAGLE,766798,LMN,とちり,S
5,2026-06-29,関西,10,abc125,TIGER,90000,DEF,あさき,S
15,2026-07-18,関東,<NA>,abc144,EAGLE,88888,ABC,いろは,S


In [7]:
# ------------------------------------------
# ダウンロードデータ (2) （システム上に登録済みのIDが収められたテーブル）
# ------------------------------------------
df_reg = pd.read_excel("reg.xlsx", skiprows=2)

con.sql("""
    DROP VIEW IF EXISTS reg;
""")

con.register("reg", df_reg)

df_reg.head(3)

,ID,Region,Label,Progress,Group,Program,Type
0,abc130,関西,いろは,100,A,Sales,Animal
1,abc131,東北,いろは,100,A,Sales,Animal
2,abc132,東海,いろは,100,A,Sales,Animal


In [8]:
# ------------------------------------------
# modified と reg をLEFT JOIN
# ------------------------------------------
con.sql("""
    DROP TABLE IF EXISTS judge;
    CREATE TEMP TABLE judge AS
        SELECT mod.*
               , CASE WHEN reg.id IS NOT NULL THEN 1 ELSE 0 END AS is_registered
        FROM
            modified mod
            LEFT JOIN (SELECT DISTINCT id FROM reg) reg
                ON mod.id = reg.id
""")

df_judge = con.sql("""
    SELECT *
    FROM judge
""").df()

df_judge['id'] = df_judge['id'].astype(str)
df_judge.to_excel("temp_judge.xlsx", sheet_name="Sheet1", index=False)

df_judge.head(3)

,datetime,region,progress,id,type,judgment_criteria,alphabet,iroha,tier,is_registered
0,2026-07-17,東海,<NA>,abc143,EAGLE,23,ABC,いろは,F,1
1,2026-07-18,関東,<NA>,abc144,EAGLE,88888,ABC,いろは,S,1
2,2026-08-07,九州,<NA>,abc164,HAWK,0,ABC,とちり,F,1


#### IDが未登録（is_registered=0）の場合
システムにアップロードしてIDを登録するためのエクセルファイルを作成する。

In [9]:
# ------------------------------------------
# アップロードデータの作成
# ------------------------------------------
con.sql("""
    DROP TABLE IF EXISTS is_registered_0;
    CREATE TEMP TABLE is_registered_0 AS
    SELECT
        id AS "ID"
        , '関東' AS "Region"
        , 'Animal' AS "Type"
        , 'create' AS "Process"
        , 'Sales' AS "Program"
        , NULL AS "Category"
        , NULL AS "Campaign"
        , NULL AS "Sales Promotion"
        , NULL AS "Start Date"
        , DATE('2028-3-31') AS "Scheduled End Date"
        , NULL AS "Manager"
        , NULL AS "Team in charge"
        , NULL AS "Label"
        , tier AS "Group"
        , NULL AS "Additional Information"
    FROM judge
    WHERE is_registered = 0
""")

df_is_registered_0 = con.sql("""
    SELECT *
    FROM is_registered_0
""").df()

df_is_registered_0['ID'] = df_is_registered_0['ID'].astype(str)

df_is_registered_0.head(3)

,ID,Region,Type,Process,Program,Category,Campaign,Sales Promotion,Start Date,Scheduled End Date,Manager,Team in charge,Label,Group,Additional Information
0,abc127,関東,Animal,create,Sales,<NA>,<NA>,<NA>,<NA>,2028-03-31,<NA>,<NA>,<NA>,F,<NA>
1,abc125,関東,Animal,create,Sales,<NA>,<NA>,<NA>,<NA>,2028-03-31,<NA>,<NA>,<NA>,S,<NA>
2,abc113,関東,Animal,create,Sales,<NA>,<NA>,<NA>,<NA>,2028-03-31,<NA>,<NA>,<NA>,F,<NA>


#### Excel保存

In [11]:
# ------------------------------------------
# Excel保存
# ------------------------------------------
# ExcelWriterを使用して、日付列の表示形式を「YYYY-MM-DD」に変更
with pd.ExcelWriter("upload_data.xlsx", engine="openpyxl") as writer:
    df_is_registered_0.to_excel(writer, sheet_name="UploadTemplate", index=False)

    # ワークシート取得
    ws = writer.sheets["UploadTemplate"]

    # ヘッダー行から対象の列番号を探す
    for cell in ws[1]:
        if cell.value == "Scheduled End Date":
            target_col = cell.column
            break

    # 2行目以降に書式適用
    for row in ws.iter_rows(
        min_row=2,
        min_col=target_col,
        max_col=target_col
    ):
        for cell in row:
            cell.number_format = "yyyy-mm-dd"

    # 列幅変更
    col_letter = get_column_letter(target_col)
    ws.column_dimensions[col_letter].width = 15

    # K～M列の幅を変更
    for col_num in range(11, 14):
        col_letter = get_column_letter(col_num)
        ws.column_dimensions[col_letter].width = 2
    
    # P～R列の幅を変更
    for col_num in range(16, 19):
        col_letter = get_column_letter(col_num)
        ws.column_dimensions[col_letter].width = 2

print('upload_data.xlsxを保存しました。')

upload_data.xlsxを保存しました。


ファイルを作成したらシステムにアップロードする。

https://～～～アップロード先～～～